This script refactors the original grid-search approach for finding optimal
ensemble weights, incorporating Bayesian Optimization with Optuna and prediction
caching for dramatically improved efficiency and scientific robustness.

0. Set Up Environment

In [1]:
!pip install albumentations torchinfo optuna
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.7 MB/s eta 0:00:00
  Cloning https://github.com/qubvel/segmentation_models.pytorch to /tmp/pip-req-build-133ibefp
  Running command git clone --filter=blob:none --quiet https://github.com/qubvel/segmentation_models.pytorch /tmp/pip-req-build-133ibefp
  Resolved https://github.com/qubvel/segmentation_models.pytorch to commit 4d20629756005085ca0f1f21605c796298ca0c16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for segmentation_models_pytorch: filename=segmentation_models_pytorch-0.5.1.dev0-py3-none-any.whl size=155806 sha256=fe33509051cfc89ea5d48e91af1e136e07a49ea0be5529ca95eec5cab640c104
  Stored in directory: /tmp/pip-ephem-wheel-cache-8y52d9d4/wheels/ef/38/4b/267c9bdb27c85ebaa11e9ec77c9059cf2c166ceee760f24429
Successfully built segmentation_models_pytorch


1. Imports

In [2]:
print("Importing libraries...")
# Install required packages if needed

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset # Added Subset
import matplotlib.pyplot as plt # Keep for potential future plots if needed
from PIL import Image, UnidentifiedImageError
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
import random
import shutil
from torch.cuda.amp import autocast
import cv2
import segmentation_models_pytorch as smp
import timm
from torchinfo import summary
import pandas as pd
import json
import warnings
import albumentations as A
from albumentations.pytorch import ToTensorV2
import zipfile
import math # For math.ceil if used later
import optuna # ### REFACTORED: Import Optuna

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

Importing libraries...
Mounted at /content/drive


In [3]:
# --- Configuration & Setup ---
print("Configuring environment...")
# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Ensemble Configuration ---
N_TOP_MODELS = 8 # Number of top models to include in the ensemble
METADATA_DIR = "/content/drive/MyDrive/METADATA_CHECKPOINTS" # Folder with _meta.json files
SORT_METRIC = "best_model_val_loss" # Metric in metadata to rank models

# ### REFACTORED: Optuna Optimization Configuration
N_OPTUNA_TRIALS = 100 # Number of weight combinations to test. 100-200 is a good starting point.
METRIC_TO_OPTIMIZE = 'dice' # The primary metric Optuna will maximize ('dice' or 'iou')

BATCH_SIZE = 64 # Can potentially use a smaller batch size for ensemble inference if memory is tight
WORKERS = 2
DEFAULT_THRESHOLD = 0.5 # Default threshold if no optimal one is found/used

# --- Dataset Source Path ---
DATASET_ZIP_DIR = '/content/drive/MyDrive/IA_MEDICA_SAMPLES/ENSEMBLE_CLEAN'

# --- Data Extraction Directory ---
base_data_dir = '/content/dataset' # Extracted fold data goes here

# --- Evaluation Configuration ---
val_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER')
val_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER_MASK')
val_not_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER')
val_not_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER_MASK')


# --- Other Config ---
DEFAULT_THRESHOLD = 0.5
OUTPUT_DIR = "/content/drive/MyDrive/RESULTS_REPORT_ENSEMBLE" # Specific output dir

DECODER_DROPOUT = 0

Configuring environment...
Using device: cuda


In [4]:
# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

Configuration complete.


In [5]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [6]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET": # <-- ADD THIS BLOCK
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights="imagenet",
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model

4. Create Custom Dataset

In [7]:
# --- 8. Dataset Class (Simplified) ---
print("Defining simplified ProstateCancerDataset...")
# Use albumentations for basic transforms (Resize, Normalize, ToTensor)
class ProstateCancerDataset(Dataset):
    def __init__(self, cancer_image_dir, cancer_mask_dir, not_cancer_image_dir, not_cancer_mask_dir):
        # Removed is_train flag as augmentations are pre-applied
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        # --- Base Transformation (Applied to ALL data) ---
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR), # Specify interpolation
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(), # Handles image normalization (scaling) and channel order (C, H, W)
                          # Converts mask to Tensor (C, H, W)
        ])

        # --- Load File Lists ---
        # Defensive listing: check if dirs exist
        self.cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
             self.cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.cancer_image_dir}")

        self.not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
             self.not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.not_cancer_image_dir}")


        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = [] # Still useful maybe for checks later

        for img_name in self.cancer_images:
             img_path = os.path.join(self.cancer_image_dir, img_name)
             mask_path = os.path.join(self.cancer_mask_dir, img_name)
             if os.path.isfile(mask_path): # Ensure mask exists
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(1)
             # else: print(f"Warning: Mask missing for cancer image {img_name}")

        for img_name in self.not_cancer_images:
             img_path = os.path.join(self.not_cancer_image_dir, img_name)
             mask_path = os.path.join(self.not_cancer_mask_dir, img_name)
             if os.path.isfile(mask_path): # Ensure mask exists
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(0)
             # else: print(f"Warning: Mask missing for non-cancer image {img_name}")

    def __len__(self):
        # Length is simply the total number of valid image/mask pairs found
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        try:
            # Load image using OpenCV (as Albumentations often uses it) - loads BGR
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None # Return None tuple on transform error


        return final_image, final_mask

print("Dataset definition complete.")

Defining simplified ProstateCancerDataset...
Dataset definition complete.


In [8]:
# --- 11. Utility Functions (Keep GPU Clear, Error Analysis, Email, Visualize) ---
print("Defining utility functions...")
def clear_gpu():
    # ... (clear_gpu remains the same) ...
    if torch.cuda.is_available():
      print("Clearing GPU cache...");
      torch.cuda.empty_cache();
      gc.collect();
      print("GPU cache cleared.");
      time.sleep(2)

Defining utility functions...


In [9]:
def load_and_select_models(metadata_dir, n_top_models, sort_metric):
    """
    Loads metadata, ranks models based on the specified metric ('best_validation_DICE'
    or 'best_model_val_loss'), and returns info for the top N.

    Args:
        metadata_dir (str): Path to the directory containing metadata JSON files.
        n_top_models (int): Number of top models to select.
        sort_metric (str): Metric to sort by. Must be either
                           'best_validation_DICE' or 'best_model_val_loss'.

    Returns:
        list: A list of dictionaries, each containing metadata for a top model.
    """
    # --- Validate sort_metric ---
    valid_sort_metrics = ["best_validation_DICE", "best_model_val_loss"]
    if sort_metric not in valid_sort_metrics:
        raise ValueError(f"Invalid sort_metric: '{sort_metric}'. Must be one of {valid_sort_metrics}")

    metadata_files = [os.path.join(metadata_dir, f) for f in os.listdir(metadata_dir) if f.endswith("_meta.json")]
    if not metadata_files:
        raise FileNotFoundError(f"No metadata files found in {metadata_dir}")

    all_metadata = []
    print(f"Loading metadata from {metadata_dir}...")
    for f_path in tqdm(metadata_files, desc="Loading Metadata"):
        try:
            with open(f_path, 'r') as f:
                meta = json.load(f)
                meta['metadata_filename'] = os.path.basename(f_path) # Add filename

                # --- Validate the sort metric value BEFORE adding ---
                metric_value = meta.get(sort_metric)
                if metric_value is not None and isinstance(metric_value, (int, float)) and not np.isnan(metric_value): # Check for NaN too
                    all_metadata.append(meta)
                else:
                    print(f"\nWarning: Skipping {os.path.basename(f_path)} - missing, non-numeric, or NaN sort metric '{sort_metric}' (value: {metric_value}).")
        except Exception as e:
            print(f"\nError loading metadata from {f_path}: {e}")

    if not all_metadata:
         raise ValueError("No valid metadata loaded after filtering for sort metric.")

    # --- Determine Sort Order based on the validated sort_metric ---
    if sort_metric == "best_validation_DICE":
        sort_descending = True # Maximize Dice
        # Default value for sorting if key is missing (shouldn't happen after filter)
        default_sort_value = -np.inf
    elif sort_metric == "best_model_val_loss":
        sort_descending = True # Minimize Loss
        default_sort_value = np.inf
    # No else needed due to initial validation

    # --- Sort the list ---
    try:
        all_metadata.sort(
            key=lambda x: x.get(sort_metric, default_sort_value), # Use validated metric
            reverse=sort_descending
        )
        print(f"\nSorted {len(all_metadata)} models by '{sort_metric}' ({'Descending' if sort_descending else 'Ascending'}).")
    except Exception as e:
        print(f"Error during sorting: {e}")
        raise

    # --- Select Top N ---
    print(f"\n--- Selecting Top {n_top_models} Models based on {sort_metric} ---")
    top_models_meta = all_metadata[:n_top_models]
    if len(top_models_meta) < n_top_models:
        print(f"Warning: Only found {len(top_models_meta)} valid models after sorting, using all of them.")
    if not top_models_meta:
         raise ValueError("No models available after sorting and selection.")

    # Print selected models
    for i, meta in enumerate(top_models_meta):
        metric_val_display = meta.get(sort_metric, 'N/A')
        try: # Format as float if possible
            metric_display_str = f"{metric_val_display:.4f}"
        except:
            metric_display_str = str(metric_val_display) # Fallback to string

        print(f" {i+1}. Arch: {meta.get('architecture', 'N/A')}, Enc: {meta.get('encoder', 'N/A')}, "
              f"{sort_metric}: {metric_display_str}, " # Use formatted string
              f"Checkpoint: {os.path.basename(meta.get('checkpoint_path', 'N/A'))}")
    print("-" * 60) # Adjusted separator width

    return top_models_meta

In [10]:
# ### REFACTORED: New function to cache predictions
def cache_predictions(models_list, dataloader, device):
    """
    Runs inference once for each model and stores predictions and ground truths.
    This avoids re-computing predictions for every trial.
    """
    print("\n--- Caching predictions from all models on the validation set ---")
    for model in models_list:
        model.eval()

    all_model_preds = [[] for _ in range(len(models_list))]
    all_true_masks = []

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Caching Batches"):
            if images is None or masks is None:
                continue
            images = images.to(device, non_blocking=True)
            # Store ground truth masks (cancer channel) on CPU to save VRAM
            true_cancer_masks = masks[:, 1, :, :].cpu()
            all_true_masks.append(true_cancer_masks)

            for i, model in enumerate(models_list):
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    outputs = model(images)
                    # Handle tuple outputs
                    if isinstance(outputs, tuple): outputs = outputs[0]
                    # Get cancer channel probabilities
                    if outputs.shape[1] == 1:
                        probs_cancer = torch.sigmoid(outputs).squeeze(1)
                    else: # Assumes 2 channels
                        probs_cancer = torch.softmax(outputs, dim=1)[:, 1, :, :]

                    # Store predictions on CPU
                    all_model_preds[i].append(probs_cancer.cpu())

    # Concatenate all batch results into single tensors
    print("Concatenating cached tensors...")
    final_preds = [torch.cat(preds) for preds in all_model_preds]
    final_trues = torch.cat(all_true_masks)

    print(f"Caching complete. Shape of true masks: {final_trues.shape}")
    print(f"Shape of predictions for one model: {final_preds[0].shape}")
    return final_preds, final_trues

In [11]:
# ### REFACTORED: Threshold function now works on cached tensors
def find_optimal_threshold_on_cached(ensemble_probs_cancer, true_masks, metric='dice', num_steps=100, smooth=1e-6):
    """
    Finds the optimal threshold on pre-computed probability maps.
    This is much faster as it doesn't require a dataloader or model inference.
    """
    thresholds = torch.linspace(0.01, 0.99, num_steps)
    best_score = -1.0
    best_threshold = 0.5

    # Move tensors to GPU for faster computation if available
    true_masks = true_masks.to(device)
    ensemble_probs_cancer = ensemble_probs_cancer.to(device)

    # Vectorized computation for speed
    tp = torch.empty(num_steps, device=device)
    fp = torch.empty(num_steps, device=device)
    fn = torch.empty(num_steps, device=device)

    for i, thresh in enumerate(tqdm(thresholds, desc="Finding Optimal Threshold", leave=False)):
        preds = (ensemble_probs_cancer > thresh).int()
        tp[i] = ((preds == 1) & (true_masks == 1)).sum()
        fp[i] = ((preds == 1) & (true_masks == 0)).sum()
        fn[i] = ((preds == 0) & (true_masks == 1)).sum()

    if metric == 'dice':
        scores = (2. * tp + smooth) / (2. * tp + fp + fn + smooth)
    elif metric == 'iou':
        scores = (tp + smooth) / (tp + fp + fn + smooth)
    else:
        raise ValueError("metric must be 'dice' or 'iou'")

    best_idx = torch.argmax(scores)
    best_score = scores[best_idx].item()
    best_threshold = thresholds[best_idx].item()

    return best_threshold, best_score

In [12]:
# ==============================================================================
# --- Main Logic ---
# ==============================================================================

# --- Setup: Data Extraction, Model Loading ---
# (This part is largely the same as the original script)
fold_zip_filename = 'MASTER_SET_1.zip'
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)

print("Extracting data...")
with zipfile.ZipFile(fold_zip_path, 'r') as z:
    z.extractall(base_data_dir)

Extracting data...


In [13]:
print("Creating DataLoader...")
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch) if batch else (None, None)

val_ds = ProstateCancerDataset(val_cancer_image_dir, val_cancer_mask_dir, val_not_cancer_image_dir, val_not_cancer_mask_dir)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True, collate_fn=collate_fn)
print(f"Validation dataset size: {len(val_ds)}")

Creating DataLoader...
Validation dataset size: 8771


In [14]:
print('Loading and Selecting Top N Models...')
top_models_metadata = load_and_select_models(METADATA_DIR, N_TOP_MODELS, SORT_METRIC)

ensemble_models = []
constituent_model_info = []
for meta in top_models_metadata:
    # (Same model loading logic as your script)
    try:
        arch, enc, chkpt_path = meta['architecture'], meta['encoder'], meta['checkpoint_path']
        model = get_model(architecture=arch, encoder=enc, validation=True)
        state_dict = torch.load(chkpt_path, map_location=device)['model_state_dict']
        model.load_state_dict(state_dict, strict=False)
        model.to(device)
        model.eval()
        ensemble_models.append(model)
        constituent_model_info.append({k: meta.get(k) for k in ['architecture', 'encoder', 'checkpoint_path', 'best_validation_DICE', 'best_model_val_loss']})
    except Exception as e:
        print(f"Error loading model {meta.get('checkpoint_path')}: {e}")

if len(ensemble_models) < 2:
    print("Error: Fewer than 2 models loaded. Exiting.")
    exit(1)

Loading and Selecting Top N Models...
Loading metadata from /content/drive/MyDrive/METADATA_CHECKPOINTS...


Loading Metadata: 100%|██████████| 12/12 [00:06<00:00,  1.83it/s]



Sorted 12 models by 'best_model_val_loss' (Descending).

--- Selecting Top 8 Models based on best_model_val_loss ---
 1. Arch: DPT, Enc: tu-vit_large_patch16_224.augreg_in21k_ft_in1k, best_model_val_loss: 0.9494, Checkpoint: DPT_tu-vit_large_patch16_224.augreg_in21k_ft_in1k_13_10_2025_19_47_57_E6_VLoss_0.3309.pth
 2. Arch: FPN, Enc: senet154, best_model_val_loss: 0.9381, Checkpoint: FPN_senet154_13_10_2025_21_58_19_E22_VLoss_0.5254.pth
 3. Arch: UPERNET, Enc: tu-hiera_large_224, best_model_val_loss: 0.9372, Checkpoint: UPERNET_tu-hiera_large_224_12_10_2025_22_48_28_E2_VLoss_0.4188.pth
 4. Arch: SWIN, Enc: tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k, best_model_val_loss: 0.9293, Checkpoint: SWIN_tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k_14_10_2025_02_37_47_E2_VLoss_0.3892.pth
 5. Arch: FPN, Enc: resnet152, best_model_val_loss: 0.9286, Checkpoint: FPN_resnet152_10_10_2025_20_00_25_E4_VLoss_0.4193.pth
 6. Arch: DEEPLABV3PLUS, Enc: tu-resnest101e, best_model_val_loss: 0.9196

model.safetensors:   0%|          | 0.00/855M [00:00<?, ?B/s]

In [15]:
# ### REFACTORED: Step 1 - Cache all model predictions
cached_preds, cached_trues = cache_predictions(ensemble_models, val_loader, device)

# Free up VRAM by deleting models after caching predictions
del ensemble_models
clear_gpu()



--- Caching predictions from all models on the validation set ---


Caching Batches: 100%|██████████| 138/138 [05:16<00:00,  2.29s/it]


Concatenating cached tensors...
Caching complete. Shape of true masks: torch.Size([8771, 224, 224])
Shape of predictions for one model: torch.Size([8771, 224, 224])
Clearing GPU cache...
GPU cache cleared.


In [16]:
# ### REFACTORED: Step 2 - Define the Optuna objective function
def objective(trial):
    """
    This function is called by Optuna for each trial.
    It suggests weights, calculates the weighted ensemble prediction,
    finds the best threshold for it, and returns the corresponding score.
    """
    # Suggest raw weights for each model
    weights_raw = [trial.suggest_float(f'w_{i}', 0.0, 1.0) for i in range(N_TOP_MODELS)]

    # Normalize weights to sum to 1
    sum_weights = sum(weights_raw)
    if sum_weights == 0: # Avoid division by zero
        # This can happen if all suggested weights are 0. Assign equal weight.
        weights_normalized = [1.0 / N_TOP_MODELS] * N_TOP_MODELS
    else:
        weights_normalized = [w / sum_weights for w in weights_raw]

    # Calculate the weighted average of cached predictions
    # This is a fast operation, no model inference needed here.
    ensemble_probs = torch.zeros_like(cached_preds[0])
    for i in range(N_TOP_MODELS):
        ensemble_probs += weights_normalized[i] * cached_preds[i]

    # For these specific weights, find the optimal threshold and the score at that threshold
    opt_threshold, opt_score = find_optimal_threshold_on_cached(
        ensemble_probs,
        cached_trues,
        metric=METRIC_TO_OPTIMIZE
    )

    # Store extra information in the trial for later analysis
    trial.set_user_attr("optimal_threshold", opt_threshold)
    trial.set_user_attr("normalized_weights", weights_normalized)

    # Return the score that Optuna should maximize
    return opt_score

In [17]:
# ### REFACTORED: Step 3 - Run the Optuna optimization study
print(f"\n--- Starting Optuna Optimization ({N_OPTUNA_TRIALS} trials) ---")

# We want to maximize the metric (e.g., Dice score)
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED) # TPESampler is the default, good for this task
)

[I 2025-10-14 04:46:31,929] A new study created in memory with name: no-name-057fbb3d-fcb7-48c7-9673-ec37b635e1d9



--- Starting Optuna Optimization (100 trials) ---


In [ ]:
# Start the optimization
study.optimize(objective, n_trials=N_OPTUNA_TRIALS)

# --- Post-Optimization Analysis ---
print("\n--- Optimization Complete ---")
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print(f"Best trial value ({METRIC_TO_OPTIMIZE}): {best_trial.value:.6f}")
print("Best trial parameters (normalized weights):")
best_weights = best_trial.user_attrs['normalized_weights']
for i, w in enumerate(best_weights):
    print(f"  Model {i+1}: {w:.4f}")
print(f"Optimal threshold for this ensemble: {best_trial.user_attrs['optimal_threshold']:.4f}")

[I 2025-10-14 04:46:53,845] Trial 0 finished with value: 0.9627283215522766 and parameters: {'w_0': 0.3745401188473625, 'w_1': 0.9507143064099162, 'w_2': 0.7319939418114051, 'w_3': 0.5986584841970366, 'w_4': 0.15601864044243652, 'w_5': 0.15599452033620265, 'w_6': 0.05808361216819946, 'w_7': 0.8661761457749352}. Best is trial 0 with value: 0.9627283215522766.
[I 2025-10-14 04:47:15,076] Trial 1 finished with value: 0.9627525210380554 and parameters: {'w_0': 0.6011150117432088, 'w_1': 0.7080725777960455, 'w_2': 0.020584494295802447, 'w_3': 0.9699098521619943, 'w_4': 0.8324426408004217, 'w_5': 0.21233911067827616, 'w_6': 0.18182496720710062, 'w_7': 0.18340450985343382}. Best is trial 1 with value: 0.9627525210380554.
[I 2025-10-14 04:47:36,310] Trial 2 finished with value: 0.9623571038246155 and parameters: {'w_0': 0.3042422429595377, 'w_1': 0.5247564316322378, 'w_2': 0.43194501864211576, 'w_3': 0.2912291401980419, 'w_4': 0.6118528947223795, 'w_5': 0.13949386065204183, 'w_6': 0.2921446485


--- Optimization Complete ---
Number of finished trials: 100
Best trial value (dice): 0.967844
Best trial parameters (normalized weights):
  Model 1: 0.3164
  Model 2: 0.1628
  Model 3: 0.1926
  Model 4: 0.1173
  Model 5: 0.0530
  Model 6: 0.1112
  Model 7: 0.0080
  Model 8: 0.0387
Optimal threshold for this ensemble: 0.4060


In [ ]:
# Save detailed results of all trials to a CSV
timestamp = get_formatted_datetime_string()
csv_results=f'CSV_results_{timestamp}.csv'

In [ ]:
# --- Post-Optimization Analysis (NEW, IMPROVED VERSION) ---
print("\n--- Optimization Complete ---")
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial
print(f"Best trial value ({METRIC_TO_OPTIMIZE}): {best_trial.value:.6f}")
print(f"Optimal threshold for this ensemble: {best_trial.user_attrs['optimal_threshold']:.4f}")



--- Optimization Complete ---
Number of finished trials: 100
Best trial value (dice): 0.967844
Optimal threshold for this ensemble: 0.4060


In [ ]:
# Save detailed results of all trials to a CSV
results_df = study.trials_dataframe()
csv_path = os.path.join(OUTPUT_DIR, csv_results)
results_df.to_csv(csv_path, index=False)
print(f"\nFull optimization results saved to: {csv_path}")


Full optimization results saved to: /content/drive/MyDrive/RESULTS_REPORT_ENSEMBLE/CSV_results_14_10_2025_05_22_06.csv


In [ ]:
# ### IMPROVEMENT: Combine weights and model info for clarity ###
best_weights = best_trial.user_attrs['normalized_weights']
constituent_models_with_weights = []
print("Best trial parameters (weights per model):")
for i, model_info in enumerate(constituent_model_info):
    weight = best_weights[i]
    model_info_with_weight = model_info.copy() # Create a copy
    model_info_with_weight['ensemble_weight'] = weight # Add the weight directly
    constituent_models_with_weights.append(model_info_with_weight)

    # Print the clear association
    arch = model_info.get('architecture', 'N/A')
    enc = model_info.get('encoder', 'N/A')
    print(f"  - Weight: {weight:.4f} -> Model: {arch} ({enc})")

Best trial parameters (weights per model):
  - Weight: 0.3164 -> Model: DPT (tu-vit_large_patch16_224.augreg_in21k_ft_in1k)
  - Weight: 0.1628 -> Model: FPN (senet154)
  - Weight: 0.1926 -> Model: UPERNET (tu-hiera_large_224)
  - Weight: 0.1173 -> Model: SWIN (tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k)
  - Weight: 0.0530 -> Model: FPN (resnet152)
  - Weight: 0.1112 -> Model: DEEPLABV3PLUS (tu-resnest101e)
  - Weight: 0.0080 -> Model: SEGFORMER (mit_b5)
  - Weight: 0.0387 -> Model: DEEPLABV3PLUS (resnet152)


In [ ]:
# Save metadata of the best ensemble configuration using the new structure
best_ensemble_meta = {
    'best_ensemble_metric': {METRIC_TO_OPTIMIZE: best_trial.value},
    'optimal_ensemble_threshold': best_trial.user_attrs['optimal_threshold'],
    # The 'ensemble_weights' key is now removed to avoid redundancy
    'constituent_models': constituent_models_with_weights, # Use the new combined list
    'optimization_details': {
        'num_trials': N_OPTUNA_TRIALS,
        'metric_optimized': METRIC_TO_OPTIMIZE,
        'datetime': timestamp
    }
}

In [ ]:
meta_path_ensemble = "ENSEMBLE_OPTIMIZATION_"+timestamp+".json"
meta_path = os.path.join(OUTPUT_DIR, meta_path_ensemble)
with open(meta_path, 'w') as f:
    json.dump(best_ensemble_meta, f, indent=4)
print(f"Best ensemble metadata saved to: {meta_path}")

Best ensemble metadata saved to: /content/drive/MyDrive/RESULTS_REPORT_ENSEMBLE/ENSEMBLE_OPTIMIZATION_14_10_2025_05_22_06.json


In [ ]:
# --- Cleanup ---
print("\nCleaning up extracted validation data...")
if os.path.exists(base_data_dir):
  try:
    shutil.rmtree(base_data_dir)
    print("Cleaned data dir.")
  except Exception as e:
    print(f"Data cleanup err: {e}")

print("\n--- Ensemble Weight Optimization Script Finished ---")


Cleaning up extracted validation data...
Cleaned data dir.

--- Ensemble Weight Optimization Script Finished ---
